In [0]:
fetch_date = dbutils.widgets.get('fetch_date')
cubeservice_table = dbutils.widgets.get('cubeservice_table')
agedtrailbalance_report_table = dbutils.widgets.get('agedtrailbalance_report_table')
bears_oblist_dim_table = dbutils.widgets.get('bears_oblist_dim_table')
client_table = dbutils.widgets.get('client_table')
office_table = dbutils.widgets.get('office_table')
bayada_customfile_src_table = dbutils.widgets.get('bayada_customfile_src_table')
date_table = dbutils.widgets.get('date_table')
payor_table = dbutils.widgets.get('payor_table')
masteraging_table = dbutils.widgets.get('masteraging_table')

In [0]:
LastDayInQtr = spark.sql(f"""
SELECT MAX(ServiceDate) AS LastDayInQtr
FROM {cubeservice_table}
WHERE ServiceQuarterNbr = (
    SELECT ServiceQuarterNbr
    FROM {cubeservice_table}
    WHERE ServiceDate = CAST('{fetch_date}' AS DATE)
)
""").collect()[0]['LastDayInQtr']

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW bears_ma_src AS
SELECT * FROM (
-- CTE 1: Deduplicate OB list
WITH tmp_dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY 
              invoice_balance_type,
              source_system_key,
              office_key,
              client_key,
              payor_key,
              date_entered_key,
              invoice_date_key,
              first_visit_date_key,
              last_visit_date_key,
              last_payment_date,
              payor_type_code,
              collector_name,
              total_days_of_service,
              invoice_number,
              net_revenue,
              orig_bill,
              account_balance,
              total_payments,
              total_adjustments,
              ar_0_90,
              ar_91_180,
              ar_181_270,
              ar_271_plus,
              npi_number
            ORDER BY invoice_number
        ) AS rnb
    FROM {bears_oblist_dim_table}
    WHERE TO_DATE(CAST(Date_Entered_Key AS STRING), 'yyyyMMdd') = CAST('{fetch_date}' AS DATE)
),
tmp AS (
    SELECT * FROM tmp_dedup WHERE rnb = 1
),
-- CTE 2: Get Client data for Invoice Number updates
client_data AS (
    SELECT ClientKey, SourceSystemId, SystemStatusCode, PayorProgramName
    FROM {client_table}
    WHERE SourceSystem = 'Bears'
),
-- CTE 3: Update Invoice Numbers for 'ADV' cases
tmp_updated AS (
    SELECT 
        tmp.*,
        CASE 
            WHEN tmp.invoice_number = 'ADV' 
            THEN CONCAT('ADV', ' - ', clt.SourceSystemId)
            ELSE tmp.invoice_number
        END AS Updated_Invoice_Number
    FROM tmp
    LEFT JOIN client_data clt ON clt.ClientKey = tmp.client_key
),
tmp1 AS (
    SELECT 
        atbf.ClientName AS Reimbursement_Team,
        atbf.Facility,
        atbf.FacilityCode AS Office_Number,
        ofc.OfficeAbbreviation AS Office_Abbreviation,
        ofc.NationalProviderIdentifier AS NPI,
        ofc.OfficeFEIN AS Tax_ID,
        ofc.Division,
        atbf.PatientType AS Practice,
        atbf.ServiceType AS State,
        atbf.MedicalRecordNumber AS Client_Number,
        atbf.Patient AS Client_Name,
        to_date(atbf.PatientDOB, 'MM/dd/yyyy') AS Client_DOB,
        clt.SystemStatusCode AS Client_Status,
        atbf.AccountNumber AS Invoice_Number,
        atbf.AccountAge AS Age_from_Last_DOS,
        DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) AS Age_From_Bill_Date,
        DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) AS Age_From_End_of_Quarter,
        to_date(atbf.LastBillDate, 'MM/dd/yyyy') AS Bill_Date,
        ob.Last_Visit_Date_Key,
        CASE 
            WHEN atbf.AccountNumber NOT LIKE '%ADV%' THEN dt.WeekendingDate 
            ELSE CAST(NULL AS DATE) 
        END AS WE_Date,
        to_date(atbf.ClaimFromDate, 'MM/dd/yyyy') AS Claim_From_Date,
        to_date(atbf.ClaimThruDate, 'MM/dd/yyyy') AS Claim_Through_Date,
        to_date(atbf.AdmitDate, 'MM/dd/yyyy') AS Admit_Date,
        to_date(atbf.DischargeDate, 'MM/dd/yyyy') AS Discharge_Date,
        NULL AS Episode_Start,
        NULL AS Episode_End,
        NULL AS Episode_ID,
        atbf.PayerCategory AS Payer_Category,
        atbf.CurrentFC AS Payer_Type,
        atbf.ActiveInsName AS Payer_Name,
        clt.PayorProgramName AS Program_Name,
        atbf.ActiveInsCode AS Active_Ins_Code_Bill_To,
        cst.BillingPeriodOrFrequency AS Billing_Frequency,
        atbf.TotalCharges AS Total_Charges,
        atbf.ExpectedNetRevenue AS Expected_Net_Revenue,
        atbf.TotalAdjustments AS Total_Adjustments,
        atbf.TotalPayments AS Total_Payments,
        atbf.AccountBalance AS Account_Balance,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 0 AND 30 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_0_30,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 31 AND 60 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_31_60,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 61 AND 90 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_61_90,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 91 AND 180 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_91_180,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 181 AND 270 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_181_270,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) >= 271 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_271_Plus,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) >= 181 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS EOQ_Aged_AR_Impact_181_Plus_Days,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), CAST('{LastDayInQtr}' AS DATE)) >= 271 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS EOQ_Aged_AR_Impact_271_Plus_Days,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 271 AND 365 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_271_365,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) BETWEEN 366 AND 540 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_366_540,
        CASE 
            WHEN DATEDIFF(DAY, to_date(atbf.LastBillDate, 'MM/dd/yyyy'), atbf._reporting_date) >= 541 
            THEN COALESCE(CAST(atbf.AccountBalance AS DOUBLE), 0.0)
        END AS Days_541_Plus,
        atbf.ClaimStatus AS Claim_Status,
        atbf.Status AS Account_Status,
        atbf.ContactType AS Contact_Type,
        atbf.AssignedTo AS Collector,
        to_date(atbf.LastAction, 'MM/dd/yyyy') AS Last_Action, 
        atbf.DaysUntouched AS Days_Untouched,
        to_date(atbf.FollowUpDate, 'MM/dd/yyyy') AS Follow_Up_Date,
        atbf.DelinquentDays AS Deliquent_Days,
        atbf.LastNote AS Last_User_Note,
        atbf.LastNoteBy AS Last_User_Note_By,
        to_date(atbf.LastNoteDate, 'MM/dd/yyyy') AS Last_User_Note_Date,
        atbf.PhysicianName AS Physician_Name,
        atbf.IsHighPriority AS Is_High_Priority,
        atbf.IsDelinquent AS Is_Delinquent,
        atbf.NumOfTouches AS Num_Of_Touches,
        atbf.RiskAssessment AS Risk_Assessment,
        'Bears' AS Source_System,
        atbf._reporting_date,
        ob.Last_Payment_Date,
        ofc.Region AS Office_Region,
        atbf.ProjectName,
        atbf.ProjectOwner,
        atbf.TrackingNumberInternal,
        atbf.TrackingNumberExternal,
        atbf.LastProjectNoteDate,
        atbf.LastProjectNoteBy,
        atbf.LastProjectNote
    FROM {agedtrailbalance_report_table} atbf
    LEFT JOIN {office_table} ofc 
        ON ofc.OfficeNumber = atbf.FacilityCode
    LEFT JOIN {bayada_customfile_src_table} cst 
        ON CAST(cst.AcctNbr AS STRING) = CAST(atbf.AccountNumber AS STRING)
    LEFT JOIN tmp_updated ob 
        ON CAST(ob.Updated_Invoice_Number AS STRING) = CAST(atbf.AccountNumber AS STRING)
    LEFT JOIN {date_table} dt 
        ON ob.Last_Visit_Date_Key = dt.DateKey
    LEFT JOIN {payor_table} py 
        ON py.PayorKey = ob.Payor_Key
    LEFT JOIN client_data clt 
        ON clt.ClientKey = ob.Client_Key
    WHERE (atbf.ActiveInsCode LIKE '%-%' 
        OR (atbf.ActiveInsCode = ' ' AND atbf.AccountNumber LIKE '%ADV%'))
    AND atbf._reporting_date = '{fetch_date}'
),
-- CTE 5: Summary balance by Client (Positive)
tmp2 AS (
    SELECT 
        MedicalRecordNumber AS PosClientNumber, 
        SUM(CAST(AccountBalance AS DOUBLE)) AS Total_Positive_Client_Balance
    FROM {agedtrailbalance_report_table}
    WHERE CAST(AccountBalance AS DOUBLE) > 0
    AND _reporting_date = '{fetch_date}'
    AND (ActiveInsCode LIKE '%-%' 
        OR (ActiveInsCode = ' ' AND AccountNumber LIKE '%ADV%'))
    GROUP BY MedicalRecordNumber
),
-- CTE 6: Summary balance by Client (Negative)
tmp3 AS (
    SELECT 
        MedicalRecordNumber AS NegClientNumber, 
        SUM(CAST(AccountBalance AS DOUBLE)) AS Total_Negative_Client_Balance
    FROM {agedtrailbalance_report_table}
    WHERE CAST(AccountBalance AS DOUBLE) < 0
    AND _reporting_date = '{fetch_date}'
    AND (ActiveInsCode LIKE '%-%' 
        OR (ActiveInsCode = ' ' AND AccountNumber LIKE '%ADV%'))
    GROUP BY MedicalRecordNumber
),
-- CTE 7: Deduplicate full rows from final join
FinalData AS (
    SELECT 
        t1.*,
        t2.Total_Positive_Client_Balance,
        t3.Total_Negative_Client_Balance,
        ROW_NUMBER() OVER (
            PARTITION BY 
                t1.Reimbursement_Team, t1.Facility, t1.Office_Number, t1.Office_Abbreviation, 
                t1.NPI, t1.Tax_ID, t1.Division, t1.Practice, t1.State, t1.Client_Number, 
                t1.Client_Name, t1.Client_DOB, t1.Client_Status, t1.Invoice_Number, 
                t1.Age_from_Last_DOS, t1.Age_From_Bill_Date, t1.Age_From_End_of_Quarter, 
                t1.Bill_Date, t1.WE_Date, t1.Claim_From_Date, t1.Claim_Through_Date, 
                t1.Admit_Date, t1.Discharge_Date, t1.Episode_Start, t1.Episode_End, 
                t1.Episode_ID, t1.Payer_Category, t1.Payer_Type, t1.Payer_Name, 
                t1.Program_Name, t1.Active_Ins_Code_Bill_To, t1.Billing_Frequency, 
                t2.Total_Positive_Client_Balance, t3.Total_Negative_Client_Balance, 
                t1.Total_Charges, t1.Expected_Net_Revenue, t1.Total_Adjustments, 
                t1.Total_Payments, t1.Account_Balance, t1.Days_0_30, t1.Days_31_60, 
                t1.Days_61_90, t1.Days_91_180, t1.Days_181_270, t1.Days_271_Plus, 
                t1.EOQ_Aged_AR_Impact_181_Plus_Days, t1.EOQ_Aged_AR_Impact_271_Plus_Days, 
                t1.Days_271_365, t1.Days_366_540, t1.Days_541_Plus, t1.Claim_Status, 
                t1.Account_Status, t1.Contact_Type, t1.Collector, t1.Last_Action, 
                t1.Days_Untouched, t1.Follow_Up_Date, t1.Deliquent_Days, t1.Last_User_Note, 
                t1.Last_User_Note_By, t1.Last_User_Note_Date, t1.Physician_Name, 
                t1.Is_High_Priority, t1.Is_Delinquent, t1.Num_Of_Touches, t1.Risk_Assessment, 
                t1.Source_System, t1._reporting_date, t1.Last_Payment_Date, t1.Office_Region, 
                t1.ProjectName, t1.ProjectOwner, t1.TrackingNumberInternal, 
                t1.TrackingNumberExternal, t1.LastProjectNoteDate, t1.LastProjectNoteBy, 
                t1.LastProjectNote
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM tmp1 t1
    LEFT JOIN tmp2 t2 ON t1.Client_Number = t2.PosClientNumber
    LEFT JOIN tmp3 t3 ON t1.Client_Number = t3.NegClientNumber
)
SELECT * FROM FinalData
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {masteraging_table} AS target
USING (
    SELECT * FROM bears_ma_src WHERE rn = 1
) AS source
ON 
    target.Invoice_Number = source.Invoice_Number AND
    target.Client_Number = source.Client_Number AND
    target.Reporting_Date = source._reporting_date

WHEN MATCHED THEN
    UPDATE SET
        target.Reimbursement_Team = source.Reimbursement_Team,
        target.Facility = source.Facility,
        target.Office_Number = source.Office_Number,
        target.Office_Abbreviation = source.Office_Abbreviation,
        target.NPI = source.NPI,
        target.Tax_ID = source.Tax_ID,
        target.Division = source.Division,
        target.Practice = source.Practice,
        target.State = source.State,
        target.Client_Name = source.Client_Name,
        target.Client_DOB = source.Client_DOB,
        target.Client_Status = source.Client_Status,
        target.Age_from_Last_DOS = source.Age_from_Last_DOS,
        target.Age_From_Bill_Date = source.Age_From_Bill_Date,
        target.Age_From_End_of_Quarter = source.Age_From_End_of_Quarter,
        target.Bill_Date = source.Bill_Date,
        target.week_ending_date = source.WE_Date,
        target.Claim_From_Date = source.Claim_From_Date,
        target.Claim_Through_Date = source.Claim_Through_Date,
        target.Admit_Date = source.Admit_Date,
        target.Discharge_Date = source.Discharge_Date,
        target.Episode_Start = source.Episode_Start,
        target.Episode_End = source.Episode_End,
        target.Episode_ID = source.Episode_ID,
        target.Payer_Category = source.Payer_Category,
        target.Payer_Type = source.Payer_Type,
        target.Payer_Name = source.Payer_Name,
        target.Program_Name = source.Program_Name,
        target.Active_Ins_Code_Bill_To = source.Active_Ins_Code_Bill_To,
        target.Billing_Frequency = source.Billing_Frequency,
        target.Total_Positive_Client_Balance = source.Total_Positive_Client_Balance,
        target.Total_Negative_Client_Balance = source.Total_Negative_Client_Balance,
        target.Total_Charges = source.Total_Charges,
        target.Expected_Net_Revenue = source.Expected_Net_Revenue,
        target.Total_Adjustments = source.Total_Adjustments,
        target.Total_Payments = source.Total_Payments,
        target.Account_Balance = source.Account_Balance,
        target.Days_0_30 = source.Days_0_30,
        target.Days_31_60 = source.Days_31_60,
        target.Days_61_90 = source.Days_61_90,
        target.Days_91_180 = source.Days_91_180,
        target.Days_181_270 = source.Days_181_270,
        target.Days_271_Plus = source.Days_271_Plus,
        target.EOQ_Aged_AR_Impact_181_Plus_Days = source.EOQ_Aged_AR_Impact_181_Plus_Days,
        target.EOQ_Aged_AR_Impact_271_Plus_Days = source.EOQ_Aged_AR_Impact_271_Plus_Days,
        target.Days_271_365 = source.Days_271_365,
        target.Days_366_540 = source.Days_366_540,
        target.Days_541_Plus = source.Days_541_Plus,
        target.Claim_Status = source.Claim_Status,
        target.Account_Status = source.Account_Status,
        target.Contact_Type = source.Contact_Type,
        target.Collector = source.Collector,
        target.Last_Action = source.Last_Action,
        target.Days_Untouched = source.Days_Untouched,
        target.Follow_Up_Date = source.Follow_Up_Date,
        target.Deliquent_Days = source.Deliquent_Days,
        target.Last_User_Note = source.Last_User_Note,
        target.Last_User_Note_By = source.Last_User_Note_By,
        target.Last_User_Note_Date = source.Last_User_Note_Date,
        target.Physician_Name = source.Physician_Name,
        target.Is_High_Priority = source.Is_High_Priority,
        target.Is_Delinquent = source.Is_Delinquent,
        target.Num_Of_Touches = source.Num_Of_Touches,
        target.Risk_Assessment = source.Risk_Assessment,
        target.Source_System = source.Source_System,
        target.Last_Payment_Date = source.Last_Payment_Date,
        target.Office_Region = source.Office_Region,
        target.project_name = source.ProjectName,
        target.project_owner = source.ProjectOwner,
        target.tracking_number_internal = source.TrackingNumberInternal,
        target.tracking_number_external = source.TrackingNumberExternal,
        target.last_project_note_date = source.LastProjectNoteDate,
        target.last_project_note_by = source.LastProjectNoteBy,
        target.last_project_note = source.LastProjectNote

WHEN NOT MATCHED THEN
    INSERT (
        Reimbursement_Team, Facility, Office_Number, Office_Abbreviation, NPI, Tax_ID, 
        Division, Practice, State, Client_Number, Client_Name, Client_DOB, Client_Status, 
        Invoice_Number, Age_from_Last_DOS, Age_From_Bill_Date, Age_From_End_of_Quarter, 
        Bill_Date, week_ending_date, Claim_From_Date, Claim_Through_Date, Admit_Date, Discharge_Date, 
        Episode_Start, Episode_End, Episode_ID, Payer_Category, Payer_Type, Payer_Name, 
        Program_Name, Active_Ins_Code_Bill_To, Billing_Frequency, Total_Positive_Client_Balance, 
        Total_Negative_Client_Balance, Total_Charges, Expected_Net_Revenue, Total_Adjustments, 
        Total_Payments, Account_Balance, Days_0_30, Days_31_60, Days_61_90, Days_91_180, 
        Days_181_270, Days_271_Plus, EOQ_Aged_AR_Impact_181_Plus_Days, 
        EOQ_Aged_AR_Impact_271_Plus_Days, Days_271_365, Days_366_540, Days_541_Plus, 
        Claim_Status, Account_Status, Contact_Type, Collector, Last_Action, Days_Untouched, 
        Follow_Up_Date, Deliquent_Days, Last_User_Note, Last_User_Note_By, Last_User_Note_Date, 
        Physician_Name, Is_High_Priority, Is_Delinquent, Num_Of_Touches, Risk_Assessment, 
        Source_System, Reporting_Date, Last_Payment_Date, Office_Region, project_name, 
        project_owner, tracking_number_internal, tracking_number_external, last_project_note_date, 
        last_project_note_by, last_project_note        
    )
    VALUES (
        source.Reimbursement_Team, source.Facility, source.Office_Number, 
        source.Office_Abbreviation, source.NPI, source.Tax_ID, source.Division, 
        source.Practice, source.State, source.Client_Number, source.Client_Name, 
        source.Client_DOB, source.Client_Status, source.Invoice_Number, 
        source.Age_from_Last_DOS, source.Age_From_Bill_Date, source.Age_From_End_of_Quarter, 
        source.Bill_Date, source.WE_Date, source.Claim_From_Date, source.Claim_Through_Date, 
        source.Admit_Date, source.Discharge_Date, source.Episode_Start, source.Episode_End, 
        source.Episode_ID, source.Payer_Category, source.Payer_Type, source.Payer_Name, 
        source.Program_Name, source.Active_Ins_Code_Bill_To, source.Billing_Frequency, 
        source.Total_Positive_Client_Balance, source.Total_Negative_Client_Balance, 
        source.Total_Charges, source.Expected_Net_Revenue, source.Total_Adjustments, 
        source.Total_Payments, source.Account_Balance, source.Days_0_30, source.Days_31_60, 
        source.Days_61_90, source.Days_91_180, source.Days_181_270, source.Days_271_Plus, 
        source.EOQ_Aged_AR_Impact_181_Plus_Days, source.EOQ_Aged_AR_Impact_271_Plus_Days, 
        source.Days_271_365, source.Days_366_540, source.Days_541_Plus, source.Claim_Status, 
        source.Account_Status, source.Contact_Type, source.Collector, source.Last_Action, 
        source.Days_Untouched, source.Follow_Up_Date, source.Deliquent_Days, 
        source.Last_User_Note, source.Last_User_Note_By, source.Last_User_Note_Date, 
        source.Physician_Name, source.Is_High_Priority, source.Is_Delinquent, 
        source.Num_Of_Touches, source.Risk_Assessment, source.Source_System, 
        source._reporting_date, source.Last_Payment_Date, source.Office_Region, 
        source.ProjectName, source.ProjectOwner, source.TrackingNumberInternal, 
        source.TrackingNumberExternal, source.LastProjectNoteDate, source.LastProjectNoteBy, 
        source.LastProjectNote
    );
""")
)